# Lab 3 — обучение MLP на CIFAR-10 в Google Colab

1. **Runtime → Change runtime type → GPU**.
2. **Рекомендуется:** `USE_TORCHVISION_DATA = True` — train и test скачиваются в Colab (~170 MB), **ничего из Kaggle/test на Drive заливать не нужно**.
3. Код: `GIT_REPO_URL` **или** zip проекта на Drive (`pack_for_colab.ps1`).
4. Kaggle `cifar-10/test/` (сотни тысяч PNG без меток) для лабы **не используется** — не загружайте его на Drive.

MLflow: `--no-mlflow`. Отчёт на ПК: `uv run python scripts/generate_report.py`.

In [ ]:
# --- Настройки ---
USE_TORCHVISION_DATA = True  # True: скачать CIFAR-10 в Colab; False: Kaggle train только с Drive
MOUNT_DRIVE = True  # False, если код только из git и бэкап скачаете через последнюю ячейку
DRIVE_ROOT = "/content/drive/MyDrive/CV-lab3"
CIFAR_ON_DRIVE = f"{DRIVE_ROOT}/cifar-10"  # только trainLabels + train PNG, без test/
PROJECT_ZIP = f"{DRIVE_ROOT}/CV-lab3-project.zip"
GIT_REPO_URL = ""
GIT_BRANCH = "main"
WORK_DIR = "/content/CV-lab3"
DRIVE_BACKUP = f"{DRIVE_ROOT}/colab_outputs"
EPOCHS_OVERRIDE = None
RUN_DEMO_PREPROCESSING = False

In [ ]:
import subprocess
import sys

def pip(*args: str) -> None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

if MOUNT_DRIVE or not GIT_REPO_URL.strip():
    from google.colab import drive

    drive.mount("/content/drive")
else:
    print("Drive не монтируем (git + torchvision data).")

In [ ]:
import os
import shutil
import subprocess
import zipfile
from pathlib import Path

work = Path(WORK_DIR)
if work.exists():
    shutil.rmtree(work)
work.mkdir(parents=True)

if GIT_REPO_URL.strip():
    subprocess.check_call(
        ["git", "clone", "--depth", "1", "-b", GIT_BRANCH, GIT_REPO_URL.strip(), str(work)]
    )
else:
    zp = Path(PROJECT_ZIP)
    if not zp.is_file():
        raise FileNotFoundError(
            f"Нет {zp}. Загрузите zip на Drive или задайте GIT_REPO_URL.\n"
            f"На Windows: .\\scripts\\pack_for_colab.ps1"
        )
    with zipfile.ZipFile(zp) as zf:
        zf.extractall(work)
    if not (work / "pyproject.toml").is_file():
        raise RuntimeError(f"В {zp} нет pyproject.toml в корне архива")

os.chdir(work)
print("Project:", work.resolve())

In [ ]:
import torch
from pathlib import Path

DATA_SOURCE = "torchvision" if USE_TORCHVISION_DATA else "kaggle"
if USE_TORCHVISION_DATA:
    print("Данные: torchvision (train+test скачаются при обучении, без Drive).")
else:
    cifar = Path(CIFAR_ON_DRIVE)
    labels = cifar / "trainLabels.csv"
    train_nested = cifar / "train" / "train"
    train_flat = cifar / "train"
    ok = labels.is_file() and (
        (train_nested.is_dir() and any(train_nested.glob("*.png")))
        or (train_flat.is_dir() and any(train_flat.glob("*.png")))
    )
    if not ok:
        raise FileNotFoundError(
            f"Нужны только Kaggle train: {labels} и PNG в train/train.\n"
            f"Папку cifar-10/test/ на Drive не кладите — она не нужна."
        )
    os.environ["CIFAR10_DATA_ROOT"] = str(cifar.resolve())
    print("CIFAR10_DATA_ROOT =", os.environ["CIFAR10_DATA_ROOT"])

print("data_source =", DATA_SOURCE)
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

In [ ]:
pip("-e", ".")

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path
from datetime import datetime

VARIANTS = [
    "configs/aug_variant_2.yaml",
    "configs/aug_variant_6.yaml",
    "configs/aug_variant_7.yaml",
]

def backup_run(tag: str) -> None:
    if not MOUNT_DRIVE:
        return
    dest = Path(DRIVE_BACKUP) / tag
    if dest.exists():
        shutil.rmtree(dest)
    dest.mkdir(parents=True, exist_ok=True)
    for name in ("checkpoints", "artifacts"):
        src = Path(WORK_DIR) / name
        if src.is_dir():
            shutil.copytree(src, dest / name)
    print("Backed up to", dest)

stamp = datetime.utcnow().strftime("%Y%m%d_%H%M%S")

for cfg in VARIANTS:
    cmd = [
        sys.executable,
        "scripts/train.py",
        "--config",
        cfg,
        "--no-mlflow",
        "--download-test",
        "--data-source",
        DATA_SOURCE,
    ]
    if EPOCHS_OVERRIDE is not None:
        cmd.extend(["--epochs", str(EPOCHS_OVERRIDE)])
    print("\n>>>", " ".join(cmd))
    subprocess.check_call(cmd, cwd=WORK_DIR)
    tag = Path(cfg).stem + "_" + stamp
    backup_run(tag)

print("\nВсе прогоны завершены.")

In [ ]:
if RUN_DEMO_PREPROCESSING:
    subprocess.check_call([sys.executable, "scripts/demo_preprocessing.py"], cwd=WORK_DIR)
    backup_run("preprocess_demo")

In [ ]:
import shutil
import zipfile
from pathlib import Path

out_zip = Path("/content/lab3_download.zip")
with zipfile.ZipFile(out_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    for folder in ("checkpoints", "artifacts"):
        root = Path(WORK_DIR) / folder
        if not root.is_dir():
            continue
        for path in root.rglob("*"):
            if path.is_file():
                zf.write(path, path.relative_to(WORK_DIR))
if MOUNT_DRIVE:
    drive_copy = Path(DRIVE_BACKUP) / f"lab3_download_{stamp}.zip"
    drive_copy.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(out_zip, drive_copy)
    print("Копия на Drive:", drive_copy)
from google.colab import files

files.download(str(out_zip))
print("Локально: распакуйте в корень репо → uv run python scripts/generate_report.py")